# 📊 EDA Case Study — Smartphones (v3/v4/v5)

**Topic:** 09 EDA · **Level:** 🟠 Intermediate · **Type:** CASE STUDY

## 📖 Flow

1. Load cleaned v3 → univariate charts
2. Fix remaining tag errors (processor_brand etc)
3. Derived flags (fast_charging_available, extended_memory)
4. Export v4 → v5 (num_cores numeric)
5. KNN impute → correlations → dummies

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
pd.set_option('display.max_columns',None)

In [2]:
df = pd.read_csv('/content/smartphone_cleaned_v3.csv')

In [3]:
df.shape

(980, 24)

In [6]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster     processor_name processor_brand  num_cores  \
0           False  Snapdragon 8 Gen2      snapdragon  Octa Core   
1           False     Snapdragon 695      snapdragon  Octa Core   
2           False        Exynos 1330          exynos  Octa Core   
3           False    Snapdragon  695      snapdragon  Octa Core   
4           False     Dimensity 1080       dimensity  Octa Core   

   processor_speed  battery_capacity  fast_charging  ram_capacity  \
0              3.2            5000.0            100          12.0   
1       

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 980 entries, 0 to 979
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand_name            980 non-null    object 
 1   model                 980 non-null    object 
 2   price                 980 non-null    int64  
 3   rating                879 non-null    float64
 4   has_5g                980 non-null    bool   
 5   has_nfc               980 non-null    bool   
 6   has_ir_blaster        980 non-null    bool   
 7   processor_name        960 non-null    object 
 8   processor_brand       960 non-null    object 
 9   num_cores             974 non-null    object 
 10  processor_speed       938 non-null    float64
 11  battery_capacity      969 non-null    float64
 12  fast_charging         980 non-null    int64  
 13  ram_capacity          980 non-null    float64
 14  internal_memory       978 non-null    float64
 15  screen_size           9

In [9]:
df.isnull().sum()

brand_name                0
model                     0
price                     0
rating                  101
has_5g                    0
has_nfc                   0
has_ir_blaster            0
processor_name           20
processor_brand          20
num_cores                 6
processor_speed          42
battery_capacity         11
fast_charging             0
ram_capacity              0
internal_memory           2
screen_size               0
refresh_rate              0
resolution                0
num_rear_cameras          0
num_front_cameras         0
os                        0
primary_camera_rear       0
primary_camera_front      4
extended_memory           0
dtype: int64

## 📖 Brand overview

```python
df['brand_name'].value_counts()
df['brand_name'].value_counts().head(10).plot(kind='bar')
```

- Brand share univariate
- Bar for top 10
- Missing value check: `.isnull().sum()`

In [11]:
# brand name
df['brand_name'].value_counts()

xiaomi       134
samsung      132
vivo         111
realme        97
oppo          88
motorola      52
apple         46
oneplus       42
poco          41
tecno         33
iqoo          32
infinix       29
huawei        16
google        14
nokia         13
honor         13
itel          10
sony           9
asus           7
nubia          6
nothing        5
lava           4
jio            4
gionee         3
micromax       3
oukitel        3
lg             3
redmi          3
letv           3
ikall          3
royole         2
doogee         2
zte            2
lenovo         2
lyf            2
sharp          1
tcl            1
cat            1
leitz          1
duoqin         1
leeco          1
blu            1
vertu          1
tesla          1
cola           1
blackview      1
Name: brand_name, dtype: int64

## 🔬 Deep Dive : Brand overview & price spread

```python
df['brand_name'].value_counts()                     # listings per brand
df['brand_name'].value_counts().head(10).plot(kind='bar')
df['price'].describe()
sns.histplot(df['price'], kde=True)
df['price'].skew()
sns.boxplot(df[df['price']<200000]['price'])
df[df['price']>200000]     # ultra-premium phones
```

- count brands — market prevalence
- bar top 10 visualize
- price heavy right skew (high-end few)
- describe tells mean vs median gap
- box ignores >200k — focus bulk range
- premium segment isolated separately
- skewed → log transform later
- outliers legit categories (foldables)
- missing rating/price count check
- layer by layer — each column its own analysis

In [16]:
# top 10 phone brands
df['brand_name'].value_counts().head(10).plot(kind='bar')

<Figure size 432x288 with 1 Axes>

## 📖 Price

```python
df['price'].describe()
sns.histplot(df['price'], kde=True)
df['price'].skew()               # positive → long tail
sns.boxplot(df[df['price'] < 200000]['price'])
df[df['price'] > 200000]         # outliers inspect
```

- Hist + kde — skew right
- Outliers: 2M+ pricing suspicious → rows check
- `boxplot` subset — outliers visibility

In [18]:
# price col
df['price'].describe()

count       980.000000
mean      32520.504082
std       39531.812669
min        3499.000000
25%       12999.000000
50%       19994.500000
75%       35491.500000
max      650000.000000
Name: price, dtype: float64

In [20]:
sns.histplot(df['price'],kde=True)

<Figure size 432x288 with 1 Axes>

In [22]:
df['price'].skew()

6.591790999665567

In [24]:
sns.boxplot(df[df['price']<200000]['price'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

In [26]:
df[df['price']>200000]

    brand_name                                   model   price  rating  \
427      vertu                   Vertu Signature Touch  650000    62.0   
478     huawei        Huawei Mate 50 RS Porsche Design  239999    81.0   
887     xiaomi  Xiaomi Redmi K20 Pro Signature Edition  480000    88.0   
951     huawei        Huawei Mate 30 RS Porsche Design  214990     NaN   

     has_5g  has_nfc  has_ir_blaster      processor_name processor_brand  \
427   False     True           False      Snapdragon 801      snapdragon   
478   False     True            True  Snapdragon 8+ Gen1      snapdragon   
887   False     True           False     Snapdragon  855      snapdragon   
951    True     True            True          Kirin  990           kirin   

     num_cores  processor_speed  battery_capacity  fast_charging  \
427  Octa Core             1.50            2275.0             -1   
478  Octa Core             3.20            4700.0             66   
887  Octa Core             2.80            4

In [27]:
df['price'].isnull().sum()

0

## 📖 Rating

```python
df['rating'].describe()
sns.histplot(df['rating'], kde=True)
sns.boxplot(df['rating'])
df['rating'].isnull().sum()
```

- Ratings narrow band, slight left skew
- Nulls — later KNN impute

## 🔬 Deep Dive : Rating distribution — core metric

```python
df['rating'].describe()
sns.histplot(df['rating'], kde=True)
df['rating'].skew()
sns.boxplot(df['rating'])
df['rating'].isnull().sum()
```

- describe: 4.x mean — healthy product ratings
- hist near 4 — slight left skew (few bad)
- box tight — low variability
- null count — handle missing
- rating = perceived quality proxy
- feature for price-benchmark analysis
- distribution normality check
- ratings clustered peak — good signal
- check brand average later
- imputation strategy roundtrip

In [28]:
# rating col

In [29]:
df['rating'].describe()

count    879.000000
mean      78.258248
std        7.402854
min       60.000000
25%       74.000000
50%       80.000000
75%       84.000000
max       89.000000
Name: rating, dtype: float64

In [30]:
sns.histplot(df['rating'],kde=True)

<Figure size 432x288 with 1 Axes>

In [31]:
df['rating'].skew()

-0.6989993034105535

In [32]:
sns.boxplot(df['rating'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

In [33]:
df['rating'].isnull().sum()

101

## 📖 Boolean features view

```python
sns.countplot(df['has_5g'])
df['has_5g'].value_counts()
```

- countplot binari dekhne ka
- 5G/NFC/IR blaster adoption

## 🔬 Deep Dive : Boolean features — countplot

```python
sns.countplot(df['has_5g'])
df['has_5g'].value_counts()
sns.countplot(df['has_nfc'])
df['has_nfc'].value_counts()
sns.countplot(df['has_ir_blaster'])
```

- countplot — bar of each boolean value
- value_counts — exact numbers
- 5G adoption → how many models
- NFC, IR blaster separate
- Booleans from sim column cleaning earlier
- insights: availability % per feature
- distribution balanced/skewed?
- cross-feature with price later
- boolean feature richness
- market standardization trend captured

In [35]:
sns.countplot(df['has_5g'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

In [36]:
df['has_5g'].value_counts()

True     549
False    431
Name: has_5g, dtype: int64

In [38]:
sns.countplot(df['has_nfc'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

In [39]:
df['has_nfc'].value_counts()

False    587
True     393
Name: has_nfc, dtype: int64

In [40]:
sns.countplot(df['has_ir_blaster'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

## 📖 Processor brand fixing

```python
df['processor_brand'] = df['processor_brand'].str.replace('sanpdragon','snapdragon')
temp_df = df[df['processor_brand'] == 'qualcomm']
df.loc[temp_df.index, 'processor_brand'] = 'snapdragon'
```

- Typo/spelling sharpen: samsung→snapdragon
- Same-brand different names → coalesce
- `loc[temp_df.index, col]` re-map
- a13 → bionic

## 🔬 Deep Dive : Fixing categorical labels

```python
df['processor_brand'] = df['processor_brand'].str.replace('sanpdragon','snapdragon')
temp_df = df[df['processor_brand'] == 'qualcomm']
df.loc[temp_df.index, 'processor_brand'] = 'snapdragon'   # consolidation
temp_df = df[df['processor_brand'] == 'a13']
df.loc[temp_df.index, 'processor_brand'] = 'bionic'
df['processor_brand'].value_counts()
```

- typo typos 'sanpdragon' → 'snapdragon'
- synonyms merge: 'qualcomm' == 'snapdragon' (same chip lineage)
- Apple 'a13' → 'bionic' consistent brand name
- str.replace + loc assign — value standardization
- value_counts after — clean buckets
- category cleaning before numeric analysis
- Lowercase uniformity
- spelling merges make groupby correct
- consolidate based on domain knowledge
- data catalog tidy

In [54]:
df['processor_brand'] = df['processor_brand'].str.replace('sanpdragon','snapdragon')
df['processor_brand'] = df['processor_brand'].str.replace('apple','bionic')
df['processor_brand'] = df['processor_brand'].str.replace('samsung','exynos')

In [51]:
temp_df = df[df['processor_brand'] == 'qualcomm']

In [52]:
df.loc[temp_df.index, 'processor_brand'] = 'snapdragon'

In [46]:
temp_df = df[df['processor_brand'] == 'a13']

In [47]:
df.loc[temp_df.index, 'processor_brand'] = 'bionic'

In [55]:
df['processor_brand'].value_counts()

snapdragon    413
helio         201
dimensity     177
exynos         50
bionic         45
unisoc         26
tiger          24
google          9
kirin           7
spreadtrum      4
sc9863a         2
fusion          1
mediatek        1
Name: processor_brand, dtype: int64

In [57]:
df[df['processor_brand'].isnull()]

    brand_name                                 model  price  rating  has_5g  \
118      tesla                        Tesla Pi Phone  69999    83.0    True   
143        jio                           Jio Phone 3   4499     NaN   False   
187      ikall                         iKall Z19 Pro   8099    60.0   False   
200    samsung                    Samsung Galaxy A13  14450    75.0   False   
307    samsung  Samsung Galaxy A13 (4GB RAM + 128GB)  14999    75.0   False   
313       itel                          itel S16 Pro   6990     NaN   False   
490    samsung                    Samsung Galaxy A15  15990    63.0   False   
523    samsung                    Samsung Galaxy F14  14990    67.0   False   
575    samsung  Samsung Galaxy A13 (6GB RAM + 128GB)  16499    78.0   False   
733      ikall                             iKall Z19   7999    61.0   False   
753      tecno                   Tecno Spark Go 2022   6249    61.0   False   
769       itel                              itel A56

## 📖 Processor speed

```python
df['num_cores'].value_counts()
df['processor_speed'].describe()
sns.displot(kind='kde', data=df, x='processor_speed')
```

- cores text → count check
- speed numeric distribution
- Skew + boxplot — outliers

## 🔬 Deep Dive : Processor speed & cores distribution

```python
df['num_cores'].value_counts()
df['processor_speed'].describe()
sns.displot(kind='kde', data=df, x='processor_speed')
df['processor_speed'].skew()
sns.boxplot(df['processor_speed'])
```

- cores mostly 8 (Octa) — modern standard
- speed GHz distribution golden — skew slightly
- kde shape — processor capability curve
- box — outlier slow chips
- describe quantiles
- correlation with price later
- speed uniform enough
- categorical cores converted to number later
- semiconductor segmentation by rank
- hardware analysis depth

In [58]:
df['num_cores'].value_counts()

Octa Core    899
Hexa Core     39
Quad Core     36
Name: num_cores, dtype: int64

In [59]:
df['processor_speed'].describe()

count    938.000000
mean       2.427217
std        0.464090
min        1.200000
25%        2.050000
50%        2.300000
75%        2.840000
max        3.220000
Name: processor_speed, dtype: float64

In [60]:
sns.displot(kind='kde',data=df,x='processor_speed')

<Figure size 360x360 with 1 Axes>

In [61]:
df['processor_speed'].skew()

0.18833557463624606

In [62]:
sns.boxplot(df['processor_speed'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

In [63]:
df

    brand_name                            model  price  rating  has_5g  \
0      oneplus                    OnePlus 11 5G  54999    89.0    True   
1      oneplus        OnePlus Nord CE 2 Lite 5G  19989    81.0    True   
2      samsung            Samsung Galaxy A14 5G  16499    75.0    True   
3     motorola             Motorola Moto G62 5G  14999    81.0    True   
4       realme               Realme 10 Pro Plus  24999    82.0    True   
..         ...                              ...    ...     ...     ...   
975   motorola       Motorola Moto Edge S30 Pro  34990    83.0    True   
976      honor                      Honor X8 5G  14990    75.0    True   
977       poco  POCO X4 GT 5G (8GB RAM + 256GB)  28990    85.0    True   
978   motorola             Motorola Moto G91 5G  19990    80.0    True   
979    samsung           Samsung Galaxy M52s 5G  24990    74.0    True   

     has_nfc  has_ir_blaster     processor_name processor_brand  num_cores  \
0       True           False  Sna

## 📖 Battery & fast charging

```python
df[df['battery_capacity'] > 7000]   # suspicious rows
def fast(row):
    return 0 if row['fast_charging'] == -1 else 1
df['fast_charging_available'] = df.apply(fast, axis=1)
```

- `-1` = no fast charging → flag 0/1
- `apply(row-based)` — custom column derive
- charging watts leftover → NaN for absent

In [64]:
df['battery_capacity'].describe()

count      969.000000
mean      4817.748194
std       1009.540054
min       1821.000000
25%       4500.000000
50%       5000.000000
75%       5000.000000
max      22000.000000
Name: battery_capacity, dtype: float64

In [65]:
sns.displot(kind='kde',data=df,x='battery_capacity')

<Figure size 360x360 with 1 Axes>

In [66]:
df[df['battery_capacity'] > 7000]

    brand_name         model  price  rating  has_5g  has_nfc  has_ir_blaster  \
391    oukitel  Oukitel WP19  29990    84.0   False     True           False   
599    oukitel  Oukitel WP21  22990    82.0   False    False           False   
843     doogee  Doogee V Max  45999    88.0    True    False           False   
966    oukitel   Oukitel WP9  25899    72.0   False     True           False   

     processor_name processor_brand  num_cores  processor_speed  \
391       Helio G95           helio  Octa Core              2.0   
599       Helio G99           helio  Octa Core              2.2   
843  Dimensity 1080       dimensity  Octa Core              2.6   
966       Helio P60           helio  Octa Core              2.0   

     battery_capacity  fast_charging  ram_capacity  internal_memory  \
391           21000.0             33           8.0            256.0   
599            9800.0             66          12.0            256.0   
843           22000.0             33          12.0

## 🔬 Deep Dive : Fast charging feature engineering

```python
df['fast_charging'].describe()
def fast(row):
    if row['fast_charging'] == -1: return 0   # absent
    else: return 1                              # present
df['fast_charging_available'] = df.apply(fast, axis=1)
df['fast_charging'] = df['fast_charging'].apply(lambda x: np.nan if x <= 0 else x)
```

- fast_charging had -1 sentinel (missing flag)
- apply row-wise → boolean available column
- -1/0 → missing → 0/absent; else 1
- clean fast_charging watts: invalid → NaN
- keep watts only for real
- new column flag usable in corr
- feature engineering: raw → usable boolean
- scale to watts actual metric
- this shapes product tier
- consistent availability semantics

In [67]:
df['fast_charging'].describe()

count    980.000000
mean      36.048980
std       35.948034
min       -1.000000
25%       15.000000
50%       30.000000
75%       65.000000
max      240.000000
Name: fast_charging, dtype: float64

In [69]:
def fast(row):

  if row['fast_charging'] == -1:
    return 0
  else:
    return 1

In [71]:
df.columns

Index(['brand_name', 'model', 'price', 'rating', 'has_5g', 'has_nfc',
       'has_ir_blaster', 'processor_name', 'processor_brand', 'num_cores',
       'processor_speed', 'battery_capacity', 'fast_charging', 'ram_capacity',
       'internal_memory', 'screen_size', 'refresh_rate', 'resolution',
       'num_rear_cameras', 'num_front_cameras', 'os', 'primary_camera_rear',
       'primary_camera_front', 'extended_memory'],
      dtype='object')

In [72]:
x = df.apply(fast,axis=1)
df.insert(12,'fast_charging_available',x)

In [75]:
df['fast_charging'] = df['fast_charging'].apply(lambda x:np.nan if x == 0 or x == -1 else x)

In [76]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster     processor_name processor_brand  num_cores  \
0           False  Snapdragon 8 Gen2      snapdragon  Octa Core   
1           False     Snapdragon 695      snapdragon  Octa Core   
2           False        Exynos 1330          exynos  Octa Core   
3           False    Snapdragon  695      snapdragon  Octa Core   
4           False     Dimensity 1080       dimensity  Octa Core   

   processor_speed  battery_capacity  fast_charging_available  fast_charging  \
0              3.2            5000.0                        1     

## 📖 RAM / internal memory

```python
temp_df = df[df['internal_memory'].isnull()]
df.loc[temp_df.index, ['ram_capacity','internal_memory']] = [[4,64],[4,64]]
```

- `loc` se known missing rows fill
- Value pairs paas — ram+memory together
- `screen_size` dist — checks

In [84]:
df['ram_capacity'].value_counts()

8.0     339
6.0     234
4.0     217
12.0     86
3.0      54
2.0      32
16.0      9
1.0       7
18.0      2
Name: ram_capacity, dtype: int64

## 🔬 Deep Dive : RAM / memory imputation

```python
df['ram_capacity'].value_counts()
temp_df = df[df['internal_memory'].isnull()]
df.loc[temp_df.index,['ram_capacity','internal_memory']] = [[4,64],[4,64]]
df['internal_memory'].value_counts()
```

- identify missing memory rows
- small set → manual fill known pairs
- loc multi-col assign
- ram/ memory hierarchy — 4/64 typical budget combo
- after: value_counts sanity
- Option: groupby median impute for bigger
- manual override reveals domain guess
- verify impact on corr
- model-ready no NaN
- ML feature consistent

In [81]:
temp_df = df[df['internal_memory'].isnull()]

In [83]:
df.loc[temp_df.index,['ram_capacity','internal_memory']] = [[4,64],[4,64]]

In [85]:
df['internal_memory'].value_counts()

128.0     523
64.0      193
256.0     157
32.0       67
512.0      22
16.0       12
1024.0      5
8.0         1
Name: internal_memory, dtype: int64

In [87]:
df['screen_size'].describe()

count    980.000000
mean       6.536765
std        0.349162
min        3.540000
25%        6.500000
50%        6.580000
75%        6.670000
max        8.030000
Name: screen_size, dtype: float64

In [88]:
sns.displot(kind='kde',data=df,x='screen_size')

<Figure size 360x360 with 1 Axes>

In [89]:
df['screen_size'].skew()

-2.11619902968816

In [90]:
sns.boxplot(df['screen_size'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

## 📖 Extended memory flag + cap

```python
df['extended_memory_available'] = df['extended_memory'].apply(lambda x: 0 if x == '0' else 1)
x = df.apply(extended_extractor, axis=1).str.split(' ').str.get(0)
df['extended_upto'] = x
def transform(text):
    if text == '1': return '1024'   # 1 GB → 1024 MB
    ...
```

- Flag: available or not
- Upto capacity extract
- Unit normalization transform

In [97]:
df['extended_memory'].value_counts()

0                       362
1 TB                    262
512 GB                  116
256 GB                  100
Not Specified            88
Memory Card (Hybrid)     30
128 GB                    9
2 TB                      6
32 GB                     3
64 GB                     3
1000 GB                   1
Name: extended_memory, dtype: int64

## 🔬 Deep Dive : Extended memory — flag + capacity

```python
df['extended_memory_available'] = df['extended_memory'].apply(lambda x: 0 if x=='0' else 1)
def extended_extractor(row):
    if row['extended_memory_available'] == 0: return 0
    return row['extended_memory'].split('up to ')[1]
df['extended_upto'] = df.apply(extended_extractor, axis=1).str.replace('\u2009',' ').str.split(' ').str.get(0)
df['extended_upto'] = df['extended_upto'].apply(transform)
```

- '0' vs TB text → boolean flag available
- extract capacity number from 'up to 1 TB'
- thin-space unicode → space normalize
- split get(0) number part
- transform maps TB→1024 (standardize GB units)
- two derived from one text col
- consistent units for numeric
- apply axis=1 row function pattern
- spec text parsing expanded
- verified via value_counts

In [95]:
df['extended_memory_available'] = df['extended_memory'].apply(lambda x:0 if x == '0' else 1)

In [106]:
def extended_extractor(row):

  if row['extended_memory_available'] == 0:
    return np.nan
  else:
    if row['extended_memory'] == '1 TB':
      return 1024
    elif row['extended_memory'] == '512 GB':
      return 512
    elif row['extended_memory'] == '256 GB':
      return 256
    elif row['extended_memory'] == 'Not Specified':
      return np.nan
    elif row['extended_memory'] == 'Memory Card (Hybrid)':
      return np.nan
    elif row['extended_memory'] == '128 GB':
      return 128
    elif row['extended_memory'] == '2 TB':
      return 2048
    elif row['extended_memory'] == '32 GB':
      return 32
    elif row['extended_memory'] == '64 GB':
      return 64
    elif row['extended_memory'] == '1000 GB':
      return 1000
    
    
    

In [108]:
def extended_extractor(row):

  if row['extended_memory_available'] == 0:
    return np.nan
  else:
    if row['extended_memory'] == 'Not Specified':
      return np.nan
    elif row['extended_memory'] == 'Memory Card (Hybrid)':
      return np.nan
    else:
      return row['extended_memory']

    
    
    

In [101]:
df['extended_memory']

0         0
1      1 TB
2      1 TB
3      1 TB
4         0
       ... 
975       0
976    1 TB
977       0
978    1 TB
979    1 TB
Name: extended_memory, Length: 980, dtype: object

In [122]:
x = df.apply(extended_extractor,axis=1).str.replace('\u2009',' ').str.split(' ').str.get(0)

In [102]:
df['extended_memory_available']

0      0
1      1
2      1
3      1
4      0
      ..
975    0
976    1
977    0
978    1
979    1
Name: extended_memory_available, Length: 980, dtype: int64

In [139]:
df['extended_upto'] = x

In [140]:
df['extended_upto'].value_counts()

1       262
512     116
256     100
128       9
2         6
32        3
64        3
1000      1
Name: extended_upto, dtype: int64

In [141]:
def transform(text):

  if text == '1':
    return '1024'
  elif text == '2':
    return '2048'
  elif text == '1000':
    return '1024'
  else:
    return text

In [142]:
df['extended_upto'] = df['extended_upto'].apply(transform)

## 📖 OS cleanup

```python
def os_transform(text):
    if 'Memory' in text: return np.nan    # wrong placement
    elif 'android' in text: return 'android'
    ...
df['os'] = df['os'].apply(os_transform)
```

- OS col me aa gaya 'Memory Card' values → NaN
- Standardize: android/ios/others
- Custom apply transforms mess

In [134]:
df['os']

0      android
1      android
2      android
3      android
4      android
        ...   
975    android
976    android
977    android
978    android
979    android
Name: os, Length: 980, dtype: object

In [132]:
def os_transform(text):

  if 'Memory' in text:
    return np.nan
  elif 'android' in text:
    return text
  elif 'ios' in text:
    return text
  else:
    return 'other'

In [136]:
df['os'] = df['os'].apply(os_transform)

In [137]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster     processor_name processor_brand  num_cores  \
0           False  Snapdragon 8 Gen2      snapdragon  Octa Core   
1           False     Snapdragon 695      snapdragon  Octa Core   
2           False        Exynos 1330          exynos  Octa Core   
3           False    Snapdragon  695      snapdragon  Octa Core   
4           False     Dimensity 1080       dimensity  Octa Core   

   processor_speed  battery_capacity  fast_charging_available  fast_charging  \
0              3.2            5000.0                        1     

In [144]:
df['extended_upto'].value_counts()

1024    263
512     116
256     100
128       9
2048      6
32        3
64        3
Name: extended_upto, dtype: int64

## 📖 Drop & correlation

```python
df.drop(columns=['processor_name','extended_memory'], inplace=True)
df.corr()['rating']
```

- Redundant cols hata post-engineering
- Rating correlations (post KNN)
- `num_cores` text → numeric (Octa Core→8)

## 🔬 Deep Dive : Drop & correlation & final export

```python
df.drop(columns=['processor_name','extended_memory'], inplace=True)
df.isnull().sum()
df.corr()['rating']           # drivers of rating
df['num_cores'] = df['num_cores'].str.replace('Octa Core','8'); ...astype(float)
new_df.to_csv('smartphone_cleaned_v5.csv', index=False)

from sklearn.impute import KNNImputer
knn = KNNImputer(n_neighbors=3, weights='distance')
x = pd.DataFrame(return_array, columns=x_df.columns).corr()['price'].reset_index()
pd.get_dummies(new_df, columns=['brand_name','processor_brand','os'], drop_first=True)
```

- drop redundant raw cols (already parsed)
- missing recount — balanced
- replacements normalize cores 'Octa Core'→8
- KNN fill remaining NaN numeric
- corr by target col — identify most influential features
- get_dummies categorical → one-hot for models
- To CSV v4/v5 — versioned clean output
- corr compare before/after imputation (merge index results)
- Ready dataset downstream ML
- Full pipeline: assess → clean → transform → export

In [152]:
df.drop(columns=['processor_name','extended_memory'],inplace=True)

In [147]:
df.isnull().sum()

brand_name                     0
model                          0
price                          0
rating                       101
has_5g                         0
has_nfc                        0
has_ir_blaster                 0
processor_name                20
processor_brand               20
num_cores                      6
processor_speed               42
battery_capacity              11
fast_charging_available        0
fast_charging                211
ram_capacity                   0
internal_memory                0
screen_size                    0
refresh_rate                   0
resolution                     0
num_rear_cameras               0
num_front_cameras              0
os                            14
primary_camera_rear            0
primary_camera_front           4
extended_memory                0
extended_memory_available      0
extended_upto                480
dtype: int64

In [150]:
df.corr()['rating']

price                        0.283504
rating                       1.000000
has_5g                       0.596087
has_nfc                      0.474754
has_ir_blaster               0.156421
processor_speed              0.628446
battery_capacity            -0.015581
fast_charging_available      0.542814
fast_charging                0.527613
ram_capacity                 0.757613
internal_memory              0.481070
screen_size                  0.298272
refresh_rate                 0.610795
num_rear_cameras             0.515531
primary_camera_rear          0.562046
extended_memory_available   -0.415265
Name: rating, dtype: float64

In [162]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 980 entries, 0 to 979
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   brand_name                 980 non-null    object 
 1   model                      980 non-null    object 
 2   price                      980 non-null    int64  
 3   rating                     879 non-null    float64
 4   has_5g                     980 non-null    bool   
 5   has_nfc                    980 non-null    bool   
 6   has_ir_blaster             980 non-null    bool   
 7   processor_brand            960 non-null    object 
 8   num_cores                  974 non-null    object 
 9   processor_speed            938 non-null    float64
 10  battery_capacity           969 non-null    float64
 11  fast_charging_available    980 non-null    int64  
 12  fast_charging              769 non-null    float64
 13  ram_capacity               980 non-null    float64

In [160]:
df['primary_camera_front'].value_counts()

16      307
8       178
32      155
5       119
12       50
13       41
20       37
10       24
50       12
60       10
44        8
40        6
2         5
7         5
24        3
25        3
10.8      3
48        2
11.1      2
0.3       1
2.1       1
Main      1
10.7      1
10.1      1
12.6      1
Name: primary_camera_front, dtype: int64

In [161]:
df['primary_camera_front'] = df['primary_camera_front'].apply(lambda x: np.nan if x == 'Main' else x).astype(float)

In [167]:
df['num_cores'].value_counts()

8    899
6     39
4     36
Name: num_cores, dtype: int64

In [166]:
df['num_cores'] = df['num_cores'].str.replace('Octa Core','8')
df['num_cores'] = df['num_cores'].str.replace('Hexa Core','6')
df['num_cores'] = df['num_cores'].str.replace('Quad Core','4')

## 📖 Export v4/v5

```python
df.to_csv('smartphone_cleaned_v4.csv', index=False)
new_df['num_cores'] = new_df['num_cores'].astype(float)
new_df.to_csv('smartphone_cleaned_v5.csv', index=False)
```

- Iterative exports — v4 → v5 evolving cleaner
- Type fixed (cores numeric)
- Pipeline manage — versions make

In [165]:
df.to_csv('smartphone_cleaned_v4.csv',index=False)

In [168]:
new_df = pd.read_csv('smartphone_cleaned_v4.csv')

In [175]:
new_df[['brand_name','model','processor_brand']]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 980 entries, 0 to 979
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   brand_name                 980 non-null    object 
 1   model                      980 non-null    object 
 2   price                      980 non-null    int64  
 3   rating                     879 non-null    float64
 4   has_5g                     980 non-null    bool   
 5   has_nfc                    980 non-null    bool   
 6   has_ir_blaster             980 non-null    bool   
 7   processor_brand            960 non-null    object 
 8   num_cores                  974 non-null    float64
 9   processor_speed            938 non-null    float64
 10  battery_capacity           969 non-null    float64
 11  fast_charging_available    980 non-null    int64  
 12  fast_charging              769 non-null    float64
 13  ram_capacity               980 non-null    float64

In [170]:
new_df['num_cores'] = new_df['num_cores'].str.replace('Octa Core','8')
new_df['num_cores'] = new_df['num_cores'].str.replace('Hexa Core','6')
new_df['num_cores'] = new_df['num_cores'].str.replace('Quad Core','4')

In [174]:
new_df['num_cores'] = new_df['num_cores'].astype(float)

In [176]:
new_df.isnull().sum()

brand_name                     0
model                          0
price                          0
rating                       101
has_5g                         0
has_nfc                        0
has_ir_blaster                 0
processor_brand               20
num_cores                      6
processor_speed               42
battery_capacity              11
fast_charging_available        0
fast_charging                211
ram_capacity                   0
internal_memory                0
screen_size                    0
refresh_rate                   0
resolution                     0
num_rear_cameras               0
num_front_cameras              4
os                            14
primary_camera_rear            0
primary_camera_front           5
extended_memory_available      0
extended_upto                480
dtype: int64

In [177]:
new_df.to_csv('smartphone_cleaned_v5.csv',index=False)

In [193]:
new_df.select_dtypes(include=['object'])

    brand_name                            model processor_brand    resolution  \
0      oneplus                    OnePlus 11 5G      snapdragon  1440 x 3216    
1      oneplus        OnePlus Nord CE 2 Lite 5G      snapdragon  1080 x 2412    
2      samsung            Samsung Galaxy A14 5G          exynos  1080 x 2408    
3     motorola             Motorola Moto G62 5G      snapdragon  1080 x 2400    
4       realme               Realme 10 Pro Plus       dimensity  1080 x 2412    
..         ...                              ...             ...           ...   
975   motorola       Motorola Moto Edge S30 Pro      snapdragon  1080 x 2460    
976      honor                      Honor X8 5G      snapdragon   720 x 1600    
977       poco  POCO X4 GT 5G (8GB RAM + 256GB)       dimensity  1080 x 2460    
978   motorola             Motorola Moto G91 5G      snapdragon  1080 x 2400    
979    samsung           Samsung Galaxy M52s 5G             NaN  1080 x 2400    

          os  
0    android

## 📖 KNN impute + correlations

```python
knn = KNNImputer(n_neighbors=3, weights='distance')
return_array = knn.fit_transform(x_df)
x.corr()['price'].reset_index()   # imputed correlations
y.merge(x, on='index')            # before vs after
```

- Kernel: NaN ko nearest neighbors avg fill
- `weights='distance'` — closer majority
- Correlations before/after merge compare
- KNN imputation info-preserving feature

In [183]:
from sklearn.impute import KNNImputer

knn = KNNImputer(n_neighbors=3,weights='distance')

return_array = knn.fit_transform(x_df)

In [190]:
x = pd.DataFrame(return_array, columns=x_df.columns).corr()['price'].reset_index()

In [191]:
y = new_df.corr()['price'].reset_index()

In [192]:
x.merge(y,on='index')

                        index   price_x   price_y
0                       price  1.000000  1.000000
1                      rating  0.343434  0.283504
2                   num_cores -0.059385 -0.048561
3             processor_speed  0.497025  0.474049
4            battery_capacity -0.164240 -0.159232
5     fast_charging_available  0.116739  0.116739
6               fast_charging  0.259294  0.277591
7                ram_capacity  0.386002  0.386002
8             internal_memory  0.557168  0.557168
9                 screen_size  0.113253  0.113253
10               refresh_rate  0.244115  0.244115
11           num_rear_cameras  0.125330  0.125330
12          num_front_cameras  0.124141  0.115228
13        primary_camera_rear  0.092095  0.092095
14       primary_camera_front  0.163230  0.162995
15  extended_memory_available -0.448628 -0.448628
16              extended_upto  0.081376  0.091945

## 📖 Dummies + save

```python
pd.get_dummies(new_df, columns=['brand_name','processor_brand','os'], drop_first=True)
new_df['brand_name'] = new_df['brand_name'].astype('category')
```

- Categorical → binary dummy (one-hot)
- `drop_first` — avoid dummy trap
- `category` dtype — memory efficient
- EDA → model-ready format

In [195]:
pd.get_dummies(new_df,columns=['brand_name','processor_brand','os'],drop_first=True)

                               model  price  rating  has_5g  has_nfc  \
0                      OnePlus 11 5G  54999    89.0    True     True   
1          OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2              Samsung Galaxy A14 5G  16499    75.0    True    False   
3               Motorola Moto G62 5G  14999    81.0    True    False   
4                 Realme 10 Pro Plus  24999    82.0    True    False   
..                               ...    ...     ...     ...      ...   
975       Motorola Moto Edge S30 Pro  34990    83.0    True    False   
976                      Honor X8 5G  14990    75.0    True    False   
977  POCO X4 GT 5G (8GB RAM + 256GB)  28990    85.0    True     True   
978             Motorola Moto G91 5G  19990    80.0    True     True   
979           Samsung Galaxy M52s 5G  24990    74.0    True    False   

     has_ir_blaster  num_cores  processor_speed  battery_capacity  \
0             False        8.0             3.20            5000.0 

In [198]:
new_df['brand_name'] = new_df['brand_name'].astype('category')

In [200]:
new_df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster processor_brand  num_cores  processor_speed  \
0           False      snapdragon        8.0              3.2   
1           False      snapdragon        8.0              2.2   
2           False          exynos        8.0              2.4   
3           False      snapdragon        8.0              2.2   
4           False       dimensity        8.0              2.6   

   battery_capacity  fast_charging_available  fast_charging  ram_capacity  \
0            5000.0                        1          100.0          12.0   
1   